# ERSP Analysis — REAL TIME (non-time-warped), GO-locked
Raw signals are loaded from network storage across three patient groups using a single format-agnostic loader that handles TRC, EDF, and H5 files (load_first_raw_in_dir). Non-neural channels are removed (filter_aux_channels), and for EL patients only electrodes with anatomical labels are kept. Signals are then rereferenced to the average of white matter contacts defined per patient (apply_wm_reref), and line noise is removed at each harmonic only if a real peak is detected, with notch strength set automatically (notch_mains_harmonics). Trial timing comes from photodiode triggers saved as TSV files per patient, and trials are kept only if the stimulus lasted at least 0.5s and the response no more than 10s, with IQR used to remove remaining outliers (collect_trials). ERSPs are computed using short-time Fourier transform and warped so each trial is split 50/50 between stimulus and post-stimulus, baseline corrected before stimulus onset (compute_ersp). White matter channels are skipped. Outputs per channel are an ERSP plot, a high-gamma heatmap sorted by trial duration (plot_hg_trials), and for clustering a raw matrix and a clean image. QC outputs are two PSDs (before and after processing) and a full recording montage with trial markers (plot_montage_overview).

For each patient, loads and preprocesses raw neural signals, then runs one or both of two parallel pipelines controlled by boolean flags.

## Processing Steps (shared for all patients)

#### 1. Data Loading
- Builds patient-specific paths (raw + prep directories)
- Loads raw signals using format-agnostic loader (TRC/EDF/H5)
- Removes auxiliary channels (ECG, DC, markers etc.)

#### 2. Channel Filtering (EL patients only)
- **SEEG patients**: keeps only channels with `_` in name (e.g. `A_L6`)
- **Grid patients** (e.g. EL044): keeps only channels matching defined prefixes with a digit (e.g. `Pa1`, `T17`, `postP3`)
- Skips patient entirely if no neural channels remain

#### 3. Preprocessing
- Saves **PSD before processing** (if flag on)
- Applies **white matter rereferencing**
- Applies **adaptive mains notch filtering**
- Saves **PSD after processing** (if flag on)

#### 4. Trial Collection
- Reads trial TSV files from `prep0`
- Applies hard duration filters: `min_stim_s=0.5`, `max_post_s=10`
- Trims outliers using IQR method
- Saves QC report and histogram


## Pipeline A — ERSP Pipeline
*Runs if `RUN_ERSP_PIPELINE=True`*

- Saves **montage overview plot** with trial onset/offset markers
- For each condition and channel:
  - Computes **ERSP** (time-frequency power map)
  - Saves **HG trial plot** (`.png`, GO-locked, sorted by response duration)
  - Saves **HG trials plot** (high-gamma, trial-by-trial heatmap)

## Pipeline B — Cluster Export
*Runs if `RUN_CLUSTER_EXPORT=True`*

- Skips non-neural, bad, and WM channels
- For each condition and channel:
  - Reuses ERSP result if Pipeline A also ran (no recomputation)
  - Saves **ERSP matrix** (`.npy`) for clustering input
  - Saves **clean ERSP image** (`.png`) for clustering input

## Outputs
| Product | Location | Pipeline |
|---|---|---|
| HG plots (**GO-locked**, sorted by response duration) | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/HG/<cond>` | A |
| ERSP matrix (**real time**, GO-locked) | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/ERSP_matrix/<cond>` | B |
| ERSP clean PNG | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/ERSP_clean/<cond>` | B |
| Report + montage | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/Report` | A |

**Everything lands in one root.** Per-channel ERSP TIFFs and the PSD overviews are
140's job and are not repeated here — they are what makes `04_ersp_LM` 264 GB.
Files are tagged `_RT_GO`, so they can never be confused with 140's `_TN` cubes.

**Not time-warped.** 140 stretches every trial onto 300 bins (0% = stimulus onset,
50% = GO). The response follows GO after a variable natural delay, and warping
spends exactly that delay. Here epochs are cut in real seconds and centred on the
GO cue (`sample_offsets`). There is **no speech-onset event in the data**, so t=0
is GO, not the moment the patient spoke.

## Imports & run controls 
(the only place you change things is here and cell 4)


In [1]:
# ============================================================
# 150_ERSP_analysis_pipeline_noTwarping.ipynb
# Cell 1 — Imports & run controls  ·  REAL-TIME (non-time-warped), GO-locked
# ============================================================
import os, glob, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from itertools import groupby
import csv

from functions import lf_io_utils as io, lf_trials as tr, lf_ersp as fe, config as cfg
from functions.config import (PAT_PATIENTS, PAT_PRESETS, EL_PATIENTS, EL_PRESETS,
                               MICROEPI_MAT_PATIENTS, MICROEPI_MAT_PRESETS, COND_ALIAS)
import LFfunctions_PDextract as LF
from LF_pd import load_patient_raw

# ------------------------------------------------------------------
# RUN CONTROLS  ← edit these before each run
# ------------------------------------------------------------------
BLOCK = "LM"

# Toggle sections
RUN_PD_EXTRACTION    = False   # trial TSVs already extracted by 140; reuse them
RUN_ERSP_PIPELINE    = True
RUN_CLUSTER_EXPORT   = True
DO_MONTAGE_PSD_PLOTS = False    # PSD/montage QC belongs to 140; not repeated here

# ------------------------------------------------------------------
# REAL-TIME (non-time-warped) SETTINGS  — the only difference from 140
# ------------------------------------------------------------------
# 140 writes TIME-NORMALISED cubes: every trial is stretched onto 300 bins with
# 0% = stimulus onset, 50% = GO. That makes trials commensurable, but the
# response follows GO after a variable delay, and warping spends that delay --
# the quantity a response-timing analysis is trying to measure.
#
# Here nothing is warped. Epochs are cut in REAL SECONDS and centred on the GO
# cue, which in these TSVs is the stimulus OFFSET (`sample_offsets`). Note there
# is no speech-onset event anywhere in the data, so t=0 is GO, not the moment
# the patient actually spoke.
RT_MODE   = "RT"
RT_ALIGN  = "go"
# Generous on purpose. Stimulus durations reach ~1.5 s, so -2.5 s keeps the whole
# stimulus visible before GO, and RT_MAX_POST_S below caps the response to match.
# Short windows that start near the event let the spectrogram's
# edge frames inflate into a spurious peak -- measured: a (0.0, 2.0) window moved
# the recovered latency of a synthetic GO-locked burst from +0.25 s to +0.54 s,
# while (-2.5, 4.0) and (-1.0, 3.0) both recovered +0.25 s.
RT_WINDOW = (-2.5, 5.0)

# Responses longer than 5 s are invalid trials. This has to match RT_WINDOW's
# post-GO reach: cfg.max_post_s is 10.0, so without this a 7 s trial would be
# KEPT by collect_trials and then silently truncated by the epoch window --
# a trial that is half-measured rather than either included or rejected.
# cfg.max_post_s itself is untouched, so 140 keeps its 10 s.
RT_MAX_POST_S = 5.0

# The CLEAN PNG is what MOBA shows when you click an electrode, so it has to be
# readable. Two fixes, both to the PICTURE only -- the .npy keeps every bin:
#   * crop above 400 Hz. The cube spans 0-500 Hz in 129 bins; the top of that range
#     carries nothing and only costs frame height.
#   * draw it wide. A square frame squashed a 129 x ~480 cube into its frequency
#     extent, turning a 1.5 s response into a narrow vertical blob.
RT_CLEAN_SHOW_HZ = 400.0
RT_CLEAN_ROWS    = int(round(RT_CLEAN_SHOW_HZ / 500.0 * 128)) + 1   # 52 of 129
RT_CLEAN_FIGSIZE = (7.5, 3.0)

# ONE output root. Everything else in the project is assumed time-normalised.
RT_SCRIPT_NAME = "05_ERSP_LM_RAWONLY_RealTime"
run_root_ersp  = os.path.join(cfg.outputs_root, RT_SCRIPT_NAME)
run_root_raw   = run_root_ersp          # HG plots and cubes share one tree

# ERSP params (from config.py)
# BASELINE, stated rather than inherited. compute_ersp's RT branch reads
# `baseline_calc_w`, NOT `baseline_w`. Passing only baseline_w (as 140 does) left
# the ERSPParams dataclass default (-0.4,-0.1) in force, so cfg.baseline_w was
# passed but dead and cfg.baseline_calc_w never arrived at all -- and the HG plot
# beside it used (-0.6,-0.1). Two baselines in one figure pair.
# Both now come from cfg.baseline_w, so the cube and the HG plot agree.
# NOTE this makes the RT cubes differ from 140's TN cubes, which still use the
# (-0.4,-0.1) default. Deliberate: changing 140 would invalidate the shipped
# 04_ersp_LM_RAWONLY tree, and the two are not comparable anyway (warped vs not).
RT_BASELINE = cfg.baseline_w          # (-0.6, -0.1) s relative to STIMULUS onset

ersp_params = fe.ERSPParams(
    nperseg=cfg.nperseg, nfft=cfg.nfft, noverlap=cfg.noverlap,
    baseline_w=RT_BASELINE, baseline_calc_w=RT_BASELINE,
    proportions=cfg.proportions,
    n_time_bins=cfg.n_time_bins, vmin=cfg.vmin, vmax=cfg.vmax, fmax=cfg.fmax
)

print(f"Controls loaded — REAL TIME, mode={RT_MODE} align={RT_ALIGN} "
      f"window={RT_WINDOW}s")
print(f"  -> {run_root_ersp}")

Controls loaded — REAL TIME, mode=RT align=go window=(-2.5, 5.0)s
  -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\05_ERSP_LM_RAWONLY_RealTime


In [2]:
# ============================================================
# Cell 2 — Helper functions
# (implementations live in lf_ersp.py and lf_io_utils.py)
# ============================================================

# Direct aliases
notch_mains_harmonics = fe.notch_mains_harmonics
fill_nans_nearest     = fe.fill_nans_nearest
save_clean_png        = fe.save_clean_png
plot_psd_overview     = fe.plot_psd_overview
_is_non_neural        = io._is_non_neural
_ensure               = io.ensure_dir

# MicroEPI .mat helpers (used for G-01..G-06 — saved as preset['pat_name'] e.g. PAT_6704)
import functions.lf_micromacro as mm

# Thin cfg-binding wrappers (keep pipeline cells unchanged)
def apply_notch_with_audit(signals, fs, patient_id, pid_raw):
    return fe.apply_notch_with_audit(
        signals, fs, patient_id, pid_raw,
        notch_patients=getattr(cfg, "notch_patients", []),
        mains_base=getattr(cfg, "mains_base", 50.0),
        fmax=getattr(cfg, "fmax", 500.0),
        repeats=getattr(cfg, "notch_repeats", 1),
        peak_z_thresh=getattr(cfg, "notch_peak_z_thresh", 3.0),
    )

def apply_wm_reref(signals, names, patient_id, *, electrodes_tsv_pattern=None):
    """Returns (signals, reref_label, wm_skip_set).

    `electrodes_tsv_pattern` overrides the auto-resolved BIDS path — used for
    MicroEPI .mat patients whose TSV lives outside the standard cohort layout.

    Raises ValueError if cfg.reref_type is not 'WM', if no WM channels are
    found for the patient, or if WM rereferencing ultimately failed to apply.
    All outputs in this pipeline MUST be WM rereferenced.
    """
    if str(cfg.reref_type).upper() != "WM":
        raise ValueError(
            f"[reref] cfg.reref_type is '{cfg.reref_type}' — only 'WM' is "
            f"allowed in this pipeline. Update config.py and re-run."
        )
    wm_idx = io.wm_indices_for_patient(patient_id, names,
                                        electrodes_tsv_pattern=electrodes_tsv_pattern)
    if not wm_idx:
        raise ValueError(
            f"[reref] {patient_id}: no WM channels found — cannot apply WM "
            f"rereferencing. Check the electrodes TSV and WM threshold."
        )
    bad = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
    signals_r, used, excluded = fe.apply_wm_reference_with_exclusions(
        signals, names, wm_idx, bad)
    if not used:
        raise ValueError(
            f"[reref] {patient_id}: WM channels were found but none were used "
            f"after exclusions — cannot guarantee WM rereferencing. "
            f"Check bad_channels_manual and available WM channels."
        )
    return signals_r, "WM", set(used) | set(excluded)


def _load_signals_and_prep_for_patient(pid_raw):
    """Route a patient to its loader.

    For MicroEPI .mat patients (G-01..G-06): load the combined macros +
    micros signal matrix via `mm.load_and_concatenate_mats` +
    `mm.build_combined_signals`. Returns the BIDS electrodes.tsv from
    `MICROEPI_MAT_PRESETS` plus the `is_micro` mask so cell 9 can apply the
    macro-only WM reref + optional anchor reref for micros.

    Everything else (PAT, EL) falls through to the standard TRC/EDF/H5
    loader and returns `is_micro=None`.

    Returns
    -------
    patient_id              : str (preset['pat_name'] for MicroEPI .mat patients)
    signals, names, fs      : as returned by the chosen loader
    prep_dir                : where collect_trials should read from
    electrodes_tsv_pattern  : explicit pattern for WM reref (or None)
    is_micro                : (n_channels,) bool mask for MicroEPI patients;
                              None for PAT / EL.
    """
    pid_str = str(pid_raw)
    if pid_str in getattr(cfg, "MICROEPI_MAT_PATIENTS", []):
        preset = cfg.MICROEPI_MAT_PRESETS[pid_str]
        patient_id = preset["pat_name"]
        d = mm.load_and_concatenate_mats(preset["data_dir"], preset["mat_files"])
        signals, names, is_micro = mm.build_combined_signals(
            d["data_ecog"], d["data_micro"], d["chans_ecog"], d["chans_micro"])
        fs = d["fs"]
        prep_dir = os.path.join(os.path.dirname(preset["data_dir"]), "prep0")
        electrodes_tsv = preset["electrodes_tsv"]
        print(f"  [paths] data_dir: {preset['data_dir']}")
        print(f"  [paths] prep_dir: {prep_dir}")
        print(f"  [microepi] {signals.shape[1]} channels = "
              f"{int((~is_micro).sum())} macros + {int(is_micro.sum())} micros")
    else:
        patient_id, raw_dir, prep_dir = io.build_paths_for_patient(pid_raw, cfg.block_name)
        signals, names, fs = io.load_first_raw_in_dir(raw_dir)
        electrodes_tsv = None  # auto-resolve from cohort
        is_micro = None
    return patient_id, signals, names, fs, prep_dir, electrodes_tsv, is_micro


print("Helpers loaded.")

Helpers loaded.


## Part 1 — Photodiode / trial extraction
Runs `LF_pd.load_patient_raw` and `LFfunctions_PDextract` for each patient in `PD_PATIENTS`.
Output: TSV timing files saved to each patient's `prep0` folder.
Set `RUN_PD_EXTRACTION = False` in Cell 1 to skip.

### PD extraction loop (PAT / EL / MicroEPI unified, calls LF_pd as-is)


In [3]:
import glob
for pid in ["EL046"]:#["EL030","EL034","EL035","EL036","EL037","EL038","EL040","EL042","EL043","EL044","EL045"]:
    new_dir = rf"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\{pid}\task_FBM\data_LM\raw"
    has_h5  = bool(glob.glob(f"{new_dir}\\*_LM.h5"))
    has_edf = bool(glob.glob(f"{new_dir}\\*_LM.edf"))
    has_tsv = bool(glob.glob(f"{new_dir}\\*.tsv"))
    print(f"  {pid}: h5={has_h5} edf={has_edf} tsv={has_tsv}  ({new_dir})")

  EL046: h5=True edf=False tsv=True  (\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL046\task_FBM\data_LM\raw)


In [4]:
import importlib; importlib.reload(cfg)
from functions import lf_io_utils as io, lf_trials as tr, lf_ersp as fe, config as cfg
from functions.config import (PAT_PATIENTS, PAT_PRESETS, EL_PATIENTS, EL_PRESETS,
                               MICROEPI_MAT_PATIENTS, MICROEPI_MAT_PRESETS, COND_ALIAS)
from LF_pd import load_patient_raw

if not RUN_PD_EXTRACTION:
    print("[skip] PD extraction (RUN_PD_EXTRACTION=False)")
else:
    # Active patients per group (empty list = skip that group)
    PAT_PATIENTS          = [] #6953
    EL_PATIENTS           = ["EL048"]  # e.g. [,"EL048","EL042","EL043","EL044","EL045"]
    MICROEPI_MAT_PATIENTS = []  # e.g. ["G-01","G-02","G-03","G-04","G-05","G-06"]

    # ────────────────────────────────────────────────────────────────────
    # PAT + EL loop — TRC / EDF / H5 loaders + standard PD detection
    # ────────────────────────────────────────────────────────────────────
    all_patients = (
        [(pid, "PAT", PAT_PRESETS) for pid in PAT_PATIENTS] +
        [(pid, "EL",  EL_PRESETS)  for pid in EL_PATIENTS]
    )

    for pid, group, presets in all_patients:
        preset = presets.get(pid)
        if preset is None:
            print(f"[skip] {pid}: no preset"); continue

        patient_id = f"PAT_{pid}" if group == "PAT" else pid
        print(f"\n=== {patient_id} ===")

        try:
            if group == "PAT":
                base_path = fr"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\{patient_id}\task_FBM\data_{BLOCK}\raw"
                raw_signals, channel_names, sampling_rate = io.load_trc_and_signals(glob.glob(os.path.join(base_path, "*.TRC"))[0])
                save_path = os.path.join(os.path.dirname(base_path), "prep0")

            elif group == "EL":
                info          = load_patient_raw(pid, block_name=BLOCK, use_el_mat_fallback=False, verbose=True)
                raw_signals   = info["raw_signals"]
                sampling_rate = info["sampling_rate"]
                channel_names = list(info["channel_names"])
                print(channel_names)
                save_path     = info["save_path"]
                exp_file      = info["matching_files_onsets"][0] if info["matching_files_onsets"] else None

            if group != "EL":
                exp_files = glob.glob(os.path.join(base_path, "*.tsv")) or glob.glob(os.path.join(base_path, "*.txt"))
                exp_file  = exp_files[0] if exp_files else None

            lower_map = {str(c).lower(): str(c) for c in channel_names}
            trig_key  = preset["trig"].lower()
            if trig_key not in lower_map:
                print(f"  [warn] trigger '{preset['trig']}' not found — skipping"); continue
            pd_name = lower_map[trig_key]

            mt = preset["manual_trig"]
            manual_path = None
            if mt:
                manual_path = mt if os.path.isabs(mt) else os.path.join(save_path, mt)
                if not os.path.exists(manual_path):
                    print(f"  [warn] manual triggers not found: {manual_path}")
                    manual_path = None

            print("FLIPS!!! ", preset["flip"])
            on_abs, off_abs, metrics = LF.get_trigger_indexes_photodiode(
                raw_signals=raw_signals, sampling_rate=sampling_rate,
                channel_names=channel_names, trig_name=pd_name,
                time_range=preset["time_range"], threshold_val=0.40,
                flip_trigs=preset["flip"],
                trial_ids=preset["trial_ids"], invalid_trials=preset["invalid_trials"],
                ignore_invalid=False, fake_trials=preset["fake_trials"],
                extra_table_path=exp_file, manual_trigs_path=manual_path,
                return_extra_metrics=True,
            )
            print(f"  Paired trials: {len(on_abs)}")
            LF.parse_and_save(pid, patient_id, on_abs, off_abs, metrics,
                              sampling_rate, save_path, BLOCK, exp_file,
                              preset["trial_ids"], preset["trig"],
                              cond_alias=COND_ALIAS)

        except Exception as e:
            print(f"[error] {patient_id}: {e}")

    # ────────────────────────────────────────────────────────────────────
    # MicroEPI .mat PD extraction (G-01..G-06, Geneva cohort)
    # Mirrors notebook 11_'s flow exactly:
    #   1) load + concat per-block .mat exports via mm.load_and_concatenate_mats
    #   2) photodiode detection on the dedicated PD trace
    #   3) merge condition_name / resp_accuracy / trial_idx from the
    #      behavioral events TSV
    #   4) write per-condition prep0 TSVs in the same format collect_trials
    #      reads downstream (so cell 9 doesn't care this is MicroEPI)
    # ────────────────────────────────────────────────────────────────────
    for pid in MICROEPI_MAT_PATIENTS:
        preset = cfg.MICROEPI_MAT_PRESETS.get(pid)
        if preset is None:
            print(f"\n[skip] MicroEPI-{pid}: no preset in cfg.MICROEPI_MAT_PRESETS"); continue
        patient_id = preset["pat_name"]
        print(f"\n=== MicroEPI-{pid} → {patient_id} ===")
        try:
            # 1) Load + concatenate per-block .mat files
            d = mm.load_and_concatenate_mats(preset["data_dir"], preset["mat_files"])
            fs = d["fs"]
            print(f"  signals: ecog {d['data_ecog'].shape}  micro {d['data_micro'].shape}  fs={fs}")

            # 2) Photodiode event detection
            beh_path = os.path.join(preset["data_dir"], preset["tsv_file"])
            on_abs, off_abs = mm.extract_events_from_photodiode(
                d["photodiode"], fs,
                time_range=preset.get("time_range", (0, -1)),
                trial_ids=preset.get("trial_ids", []) or None,
                invalid_trials=preset.get("invalid_trials", []) or None,
                fake_trials=preset.get("fake_trials", []) or None,
                extra_table_path=beh_path,
                do_plot=False,
            )
            print(f"  photodiode: {len(on_abs)} onsets, {len(off_abs)} offsets")

            # 3) Pull condition_name / resp_accuracy / trial_idx from the behavioral TSV
            beh = LF._read_trial_table(beh_path)
            dfl = beh["raw_df"].rename(columns=str.lower)
            def _pick(cols):
                return next((dfl[c].astype(str).to_numpy() for c in cols if c in dfl), None)
            condition_name = _pick(["category", "blockname"])
            resp_accuracy  = _pick(["response_type", "responseaccuracy"])
            trial_idx_col  = _pick(["exemplar", "stimnumber"])
            trial_ids_for_save = (preset.get("trial_ids") or
                                  ([str(x).lower() for x in condition_name]
                                   if condition_name is not None else []))

            # 4) Write per-condition prep0 TSVs (same format as PAT/EL)
            prep_dir = os.path.join(os.path.dirname(preset["data_dir"]), "prep0")
            LF.save_onsets_offsets_by_condition(
                patient_id=patient_id, block_name=BLOCK,
                onsets=on_abs, offsets=off_abs, sampling_rate=fs,
                trial_ids=trial_ids_for_save, out_dir=prep_dir,
                condition_name=condition_name, resp_accuracy=resp_accuracy,
                trial_idx=trial_idx_col,
                cond_alias=COND_ALIAS,
                trigger_label=preset.get("trig", "photodiode"),
            )
            print(f"  [{patient_id}] PD extraction done -> {prep_dir}")
        except Exception as e:
            print(f"[error] MicroEPI {pid}: {e}")

    print("\n[PD extraction done]")

[skip] PD extraction (RUN_PD_EXTRACTION=False)


## Part 2 — ERSP pipeline + cluster export
Reads raw data and `prep0` TSVs. Toggles:
- `RUN_ERSP_PIPELINE = True` → GO-locked HG plots, Report, montage QC into `05_ERSP_LM_RAWONLY_RealTime/`
- `RUN_CLUSTER_EXPORT = True` → per-channel `ERSP_matrix/*_RT_GO.npy` and `ERSP_clean/*.png` into the same root (consumed by `05_FBM_ResponseTiming`, **not** by `02_FBM_Clustering`, which expects 300 warped bins)

Both flags can be on simultaneously — the loop computes the ERSP once per (channel, condition) and dispatches outputs to whichever pipeline is enabled.

In [5]:
# cfg.patient_ids=["EL034","EL037", "EL038", "EL040", "EL045","EL044","EL030","EL033","EL048"]
# cfg.patient_ids=["EL044","EL033","EL048"] 
# cfg.patient_ids=[2868, 3066, 3390, 3415, 3455, 3965, 3975, 3780]

In [6]:
# Cohort for the real-time run: 6 MicroEPI + 14 Bern + 10 HUG = 30.
cfg.patient_ids = ["G-06", "G-04", "G-05", "G-01", "G-02", "G-03",
                   "EL030", "EL033", "EL034", "EL035", 
                   "EL036", "EL037", "EL038",
                   "EL040", "EL042", "EL043", "EL044", "EL045", "EL046", "EL048",
                   "PAT_3455", "PAT_2868", "PAT_3066", "PAT_3301", "PAT_3390",
                   "PAT_3415", "PAT_3965", "PAT_3975", "PAT_3780", "PAT_6953"]
#assert len(cfg.patient_ids) == len(set(cfg.patient_ids)) == 30
print(f"cohort: {len(cfg.patient_ids)} patients")
# PAT_6953 is the one patient with raw but no extracted trials -- it will be
# skipped with "no trials" while RUN_PD_EXTRACTION is False.
# cfg.patient_ids=["G-03"]


cohort: 30 patients


In [7]:
import gc, os
import pandas as pd

# ──────────────────────────────────────────────────────────────────────────
# Per-cell toggles
# ──────────────────────────────────────────────────────────────────────────
# When False, MicroEPI micros are skipped in the per-channel ERSP/HG/cluster
# loop — only macros are processed (matches the pre-refactor behaviour).
# When True (default), micros are treated like any other channel: ERSP/HG
# plots get saved and the cluster export writes one .npy per micro.
INCLUDE_MICROEPI_MICROS = True

if not RUN_ERSP_PIPELINE and not RUN_CLUSTER_EXPORT:
    print("[skip] both pipelines disabled")
    # Stubs so the per-patient cells below fail with an explanation rather than
    # a bare NameError.
    def run_patients(ids):
        print("[skip] set RUN_ERSP_PIPELINE / RUN_CLUSTER_EXPORT and re-run this cell")
    def wm_report():
        print("[skip] nothing was run")
else:
    # Per-patient summary collected during the loop and printed/saved at the end.
    # Status codes: ok | ok-no-trials | error-load | error-reref | error-other
    wm_report_rows = []

    def process_patient(pid_raw):
        """
        Process one patient end-to-end. Wrapped in a function so all heavy
        intermediates (raw signals, ERSP cubes, matplotlib figures) become
        garbage-collectable on return — keeps RAM bounded across 16+ patients
        without needing a kernel restart.
        """
        report = {
            "pid_raw": str(pid_raw),
            "patient_id": "",
            "status": "",
            "n_channels_in": 0,
            "n_channels_neural": 0,
            "n_channels_unknown_dropped": 0,
            "n_channels_used": 0,
            "n_wm_used": 0,
            "wm_channels_used": "",
            "wm_channels_excluded_as_bad": "",
            "error": "",
        }
        try:
            patient_id, signals, names, fs, prep_dir, _wm_tsv, is_micro = \
                _load_signals_and_prep_for_patient(pid_raw)
            report["patient_id"] = patient_id
            report["n_channels_in"] = len(names)
            is_microepi = is_micro is not None

            # ── Aux drop (ECG / DC / markers) — applies to all cohorts.
            # For MicroEPI the loader already returns curated macros + micros,
            # so this is usually a no-op but kept for symmetry / safety.
            if is_microepi:
                # filter_aux_channels returns new arrays — keep is_micro aligned
                _n_before = len(names)
                kept_idx = [i for i, nm in enumerate(names) if not _is_non_neural(nm)]
                signals  = signals[:, kept_idx]
                names    = [names[i] for i in kept_idx]
                is_micro = is_micro[kept_idx]
                if len(names) < _n_before:
                    print(f"  [{patient_id}] aux drop: {_n_before} → {len(names)} channels")
            else:
                signals, names, *_ = io.filter_aux_channels(signals, names)

            # ── EL prefix / grid filter — EL ONLY (no MicroEPI).
            if (not is_microepi) and str(pid_raw).startswith("EL"):
                if pid_raw in cfg.EL_GRID_PATIENTS:
                    prefixes = cfg.EL_GRID_KEEP_PREFIXES.get(pid_raw, ())
                    keep = [i for i, nm in enumerate(names)
                            if any(str(nm).startswith(p) for p in prefixes)
                            and any(c.isdigit() for c in str(nm))]
                else:
                    keep = [i for i, nm in enumerate(names) if ("_" in str(nm) or "-" in str(nm))]
                signals = signals[:, keep]
                names   = [names[i] for i in keep]
                if len(names) == 0:
                    print(f"[skip] {patient_id}: no neural channels")
                    report["status"] = "no-neural-channels"
                    return report
            report["n_channels_neural"] = len(names)

            # ── EXPERIMENT-WINDOW CROP (EL ONLY) ─────────────────────────
            # MicroEPI doesn't crop here — PD detection already used the
            # preset's time_range, and the .mat exports are already trimmed
            # to the recording session.
            crop_offset = 0
            _preset = cfg.EL_PRESETS.get(pid_raw) if (not is_microepi and str(pid_raw).startswith("EL")) else None
            if _preset and _preset.get("time_range") and _preset["time_range"][1] > 0:
                t0, t1 = _preset["time_range"]
                s0 = max(0, int(t0 * fs))
                s1 = min(signals.shape[0], int(t1 * fs))
                if s1 > s0 and (s1 - s0) < signals.shape[0]:
                    full_s = signals.shape[0] / fs
                    signals     = np.ascontiguousarray(signals[s0:s1, :])
                    crop_offset = s0
                    print(f"  [{patient_id}] cropped to {t0:.0f}s..{t1:.0f}s "
                          f"({(s1-s0)/fs:.1f}s of {full_s:.1f}s)")

            # ── PER-PATIENT CHANNEL-NAME OVERRIDE (EL ONLY) ──────────────
            if (not is_microepi) and patient_id in getattr(cfg, "STRIP_HEMI_PATIENTS", set()):
                import re as _re
                _hemi_re = _re.compile(r"_(?:[LR])(?=\d)")
                names = [_hemi_re.sub("", str(nm)) for nm in names]
                print(f"  [{patient_id}] stripped _L#/_R# per cfg.STRIP_HEMI_PATIENTS  "
                      f"-> first few: {names[:8]}")

            # ── DROP "UNKNOWN" PARCELLATION CHANNELS (EL/PAT ONLY) ───────
            # Skipped for MicroEPI: micros aren't in BIDS, and macros are
            # already explicitly enumerated in the .mat exports we trust.
            if not is_microepi:
                try:
                    unk_idx = set(io.unknown_indices_for_patient(
                        patient_id, names, electrodes_tsv_pattern=_wm_tsv,
                    ))
                except Exception as _e_unk:
                    print(f"[warn] {patient_id}: Unknown-channel lookup failed ({_e_unk}); keeping all")
                    unk_idx = set()
                if patient_id in getattr(cfg, "MIXED_GRID_DEPTH_PATIENTS", set()):
                    keep_prefixes = cfg.MIXED_GRID_KEEP_PREFIXES.get(patient_id, ())
                    if keep_prefixes:
                        protected = {i for i in unk_idx
                                     if any(str(names[i]).startswith(p) for p in keep_prefixes)}
                        if protected:
                            print(f"  [{patient_id}] protected {len(protected)} grid channels "
                                  f"from Unknown drop (prefixes={keep_prefixes})")
                        unk_idx -= protected
                if unk_idx:
                    report["n_channels_unknown_dropped"] = len(unk_idx)
                    keep = [i for i in range(len(names)) if i not in unk_idx]
                    signals = signals[:, keep]
                    names   = [names[i] for i in keep]
                    print(f"  [{patient_id}] dropped {len(unk_idx)} 'Unknown' channels (no parcellation in TSV)")

            # ── REREFERENCING ────────────────────────────────────────────
            # MicroEPI:  WM reref on macros only via mm.apply_wm_reref_selective.
            #            Micros optionally re-referenced to a single anchor
            #            electrode (preset['micro_reref_anchor']); otherwise
            #            left raw.
            # EL grid (cfg.EL_GRID_PATIENTS) without WM: reref='NONE' fallback.
            # Everything else: standard apply_wm_reref via lf_ersp.
            if is_microepi:
                pid_str = str(pid_raw)
                preset  = cfg.MICROEPI_MAT_PRESETS.get(pid_str, {})
                wm_names_raw = mm.derive_wm_channels_from_electrodes_tsv(_wm_tsv)
                print(f"  [{patient_id}] WM channels from TSV: {len(wm_names_raw)} "
                      f"-> {wm_names_raw[:6]}{'...' if len(wm_names_raw) > 6 else ''}")
                try:
                    signals, wm_used, wm_excl = mm.apply_wm_reref_selective(
                        signals, names, wm_names_raw, is_micro,
                        apply_wm_to_micros=False)
                except Exception as _e_reref:
                    print(f"[error] {patient_id}: macro WM reref failed — {_e_reref}")
                    report["status"] = "error-reref"
                    report["error"]  = str(_e_reref)
                    return report
                reref   = "WM"
                wm_skip = set()  # macros that ARE WM stay in the loop; selective reref doesn't zero them
                report["n_wm_used"]                   = len(wm_used)
                report["wm_channels_used"]            = "|".join(sorted(wm_used))
                report["wm_channels_excluded_as_bad"] = ""

                # Optional anchor reref for micros
                anchor_name = preset.get("micro_reref_anchor")
                signals, anchor_used = mm.apply_micro_anchor_reref(
                    signals, names, is_micro, anchor_name)
                if anchor_used:
                    print(f"  [{patient_id}] micros re-referenced to anchor: {anchor_used}")
                else:
                    print(f"  [{patient_id}] micros left raw (no micro_reref_anchor in preset)")
            else:
                is_grid    = str(pid_raw) in getattr(cfg, "EL_GRID_PATIENTS", set())
                wm_all     = io.wm_labels_for_patient(patient_id, electrodes_tsv_pattern=_wm_tsv)
                _bad_norm_for_report = {io.normalize_label(b)
                                        for b in getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])}
                _name_norm = [io.normalize_label(n) for n in names]
                wm_in_sig  = [n for n in wm_all if n in _name_norm]
                wm_usable  = [n for n in wm_in_sig if n not in _bad_norm_for_report]

                if wm_usable:
                    try:
                        signals, reref, wm_skip = apply_wm_reref(
                            signals, names, patient_id, electrodes_tsv_pattern=_wm_tsv,
                        )
                        wm_excluded_norm = [n for n in wm_in_sig if n in _bad_norm_for_report]
                        report["n_wm_used"]                    = len(wm_usable)
                        report["wm_channels_used"]             = "|".join(sorted(wm_usable))
                        report["wm_channels_excluded_as_bad"]  = "|".join(sorted(wm_excluded_norm))
                    except ValueError as _e_reref:
                        print(f"[error] {patient_id}: WM reref failed — {_e_reref}")
                        report["status"] = "error-reref"
                        report["error"]  = str(_e_reref)
                        return report
                elif is_grid:
                    # Grid patient without WM contacts. If listed in
                    # cfg.GRID_CAR_PATIENTS, CAR each array separately;
                    # otherwise fall back to no rereferencing.
                    if str(pid_raw) in getattr(cfg, "GRID_CAR_PATIENTS", set()):
                        _bad_car = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
                        _car_prefixes = cfg.EL_GRID_KEEP_PREFIXES.get(pid_raw, None)
                        signals, car_groups = fe.apply_grid_car(
                            signals, names,
                            group_prefixes=_car_prefixes,
                            bad_channels=_bad_car, min_group=2)
                        reref   = "CAR"
                        wm_skip = set()
                        _summary = ", ".join(f"{k}({len(v)})" for k, v in sorted(car_groups.items()))
                        print(f"[note] {patient_id}: per-grid CAR applied — groups: {_summary}")
                        report["n_wm_used"]                   = 0
                        report["wm_channels_used"]            = "CAR:" + _summary
                        report["wm_channels_excluded_as_bad"] = "|".join(sorted(_bad_car))
                        # not a failure — overwritten to 'ok' at end if pipeline completes
                    else:
                        print(f"[note] {patient_id}: grid patient with no WM contacts — "
                              f"falling back to reref='NONE' (no rereferencing applied)")
                        reref   = "NONE"
                        wm_skip = set()
                        report["n_wm_used"]                   = 0
                        report["wm_channels_used"]            = ""
                        report["wm_channels_excluded_as_bad"] = ""
                        report["status"]                      = "ok-no-wm-grid"
                else:
                    msg = f"no WM channels available for {patient_id} (not a grid patient)"
                    print(f"[error] {msg}")
                    report["status"] = "error-reref"
                    report["error"]  = msg
                    return report

            signals = apply_notch_with_audit(signals, fs, patient_id, pid_raw)

            if RUN_ERSP_PIPELINE and DO_MONTAGE_PSD_PLOTS:
                fe.plot_psd_overview(
                    signals=signals, fs=fs, names=names,
                    save_root=io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "PSD_clean"),
                    patient_id=patient_id, block_name=cfg.block_name,
                    fmax=cfg.fmax, mains_base=getattr(cfg, "mains_base", 50.0), dpi=600,
                )

            report_dir  = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "Report")
            report_path = os.path.join(report_dir, f"{patient_id}_IQR.tsv")
            cond_groups = tr.collect_trials(prep_dir, fs, outlier_method="IQR",
                                            iqr_k=cfg.iqr_k, report_path=report_path,
                                            patient_id=patient_id, max_post_s=RT_MAX_POST_S,
                                            condition_aliases=cfg.COND_ALIAS)
            if not cond_groups:
                print(f"[skip] {patient_id}: no trials")
                report["status"] = "ok-no-trials"
                return report

            # Re-base trigger sample indices into the cropped signal.
            # Only EL patients are cropped (crop_offset > 0); MicroEPI / PAT
            # never enter this branch.
            if crop_offset > 0:
                rebased = {}
                n_samples = signals.shape[0]
                for cond, (on, off, tend) in cond_groups.items():
                    on   = np.asarray(on)   - crop_offset
                    off  = np.asarray(off)  - crop_offset
                    tend = np.asarray(tend) - crop_offset
                    ok = (on >= 0) & (tend <= n_samples)
                    if (~ok).any():
                        print(f"  [{patient_id} | {cond}] dropped {(~ok).sum()} "
                              f"trials outside crop window")
                    rebased[cond] = (on[ok], off[ok], tend[ok])
                cond_groups = rebased

            if RUN_ERSP_PIPELINE:
                tr.plot_montage_overview(
                    signals=signals, fs=fs, names=names,
                    cond_groups=cond_groups, save_dir=report_dir, patient_id=patient_id,
                    fmt="png",     # was TIFF: 9 files, ~1 GB each, 73% of the tree
                )
                hg_root   = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "HG")

            if RUN_CLUSTER_EXPORT:
                mat_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_matrix"))
                img_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_clean"))
                _bad_norm = {io.normalize_label(b)
                             for b in getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])}
                skip      = set(n for n in names if _is_non_neural(n))
                skip     |= {n for n in names if io.normalize_label(n) in _bad_norm}
                skip     |= wm_skip

            # Build the set of micro channel names (used to optionally skip
            # micros in the per-channel loop when INCLUDE_MICROEPI_MICROS is
            # False). For non-MicroEPI patients this is always empty.
            micro_names_set = set()
            if is_microepi and not INCLUDE_MICROEPI_MICROS:
                micro_names_set = {names[i] for i in range(len(names)) if is_micro[i]}

            for cond, (onsets, offsets, trial_ends) in cond_groups.items():
                if RUN_ERSP_PIPELINE:
                    hg_dir   = _ensure(os.path.join(hg_root, cond))
                if RUN_CLUSTER_EXPORT:
                    out_mat  = _ensure(os.path.join(mat_root, cond))
                    out_png  = _ensure(os.path.join(img_root, cond))

                print(f"  {cond}: ", end="", flush=True)
                for ci, chan_name in enumerate(names):
                    if chan_name in wm_skip: continue
                    if chan_name in micro_names_set: continue
                    if RUN_CLUSTER_EXPORT and chan_name in skip and not RUN_ERSP_PIPELINE: continue

                    res = fe.compute_ersp(
                        signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                        trial_ends=trial_ends, mode=RT_MODE, time_window=RT_WINDOW,
                        align=RT_ALIGN, params=ersp_params,
                    )

                    if RUN_ERSP_PIPELINE:
                        # No per-channel ERSP TIFF here. That is 140's job and it is
                        # what makes 04_ersp_LM 264 GB; this tree carries only the
                        # HG trial plots and the cubes.
                        fe.plot_hg_trials(
                            signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                            chan_name=chan_name, patient_id=patient_id, condition=cond, reref_type=reref,
                            time_window=RT_WINDOW, baseline_w=RT_BASELINE,
                            hg_band=cfg.hg_band, smooth_ms=cfg.hg_smooth_ms,
                            vmin=cfg.hg_vmin, vmax=cfg.hg_vmax,
                            save_dir=hg_dir, add_separators=False, sort_ascending=True,
                            trial_end_indices=trial_ends,
                            # t=0 is GO; rows ordered by how long the response lasted,
                            # so the green trial-end ticks form a staircase.
                            sort_by="resp", align="go", fmt="png",
                        )

                    if RUN_CLUSTER_EXPORT and chan_name not in skip:
                        A = np.array(res["avg_db"], float)
                        if np.isnan(A).any():
                            print(f"[warn] {patient_id} {cond} {chan_name}: {int(np.isnan(A).sum())} NaNs → filling")
                            fill_nans_nearest(A)
                        # 140 tags TN and leaves RT untagged, which would make these
                        # indistinguishable from legacy untagged files. Tag both.
                        _m = str(res["meta"]["mode"]).upper()
                        mode_tag = "_TN" if _m == "TN" else (
                            "_RT_GO" if res["meta"].get("align") == "go" else "_RT")
                        stem = f"{patient_id}_{cond}_{reref}_ERSP_{chan_name}{mode_tag}"
                        np.save(os.path.join(out_mat, f"{stem}.npy"), A)
                        save_clean_png(A, vmin=ersp_params.vmin, vmax=ersp_params.vmax,
                                       path_png=os.path.join(out_png, f"{stem}_CLEAN.png"),
                                       keep_rows=RT_CLEAN_ROWS, figsize=RT_CLEAN_FIGSIZE)
                        del A
                    del res
                print(" done")

            # Preserve the "ok-no-wm-grid" tag if we set it earlier;
            # otherwise this is a fully-normal "ok".
            if not report["status"]:
                report["status"] = "ok"
            return report
        except Exception as e:
            print(f"[error] {pid_raw}: {e}")
            report["status"] = report["status"] or "error-other"
            report["error"]  = str(e)
            return report

    # ────────────────────────────────────────────────────────────────────────
    # Main loop. Per-patient bodies run inside process_patient(); we then
    # explicitly close all matplotlib figures + GC to keep memory bounded.
    # ────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────
    # Per-patient driver. One patient per cell below, so any single patient can
    # be re-run on its own without touching the others. Re-running a patient
    # REPLACES its row in the report rather than appending a second one.
    # ────────────────────────────────────────────────────────────────────────
    def _row_pid(row):
        for k in row:
            if "patient" in k.lower():
                return str(row[k])
        return None

    def run_patients(ids):
        """Process one patient (or a few). Safe to call repeatedly."""
        if isinstance(ids, str):
            ids = [ids]
        last = None
        for pid_raw in ids:
            print("\n"); print(pid_raw)
            row = process_patient(pid_raw)
            pid = _row_pid(row)
            if pid is not None:
                wm_report_rows[:] = [r for r in wm_report_rows if _row_pid(r) != pid]
            wm_report_rows.append(row)
            last = row
            plt.close("all"); gc.collect()
        return last

    def wm_report():
        """Print and save the report over whatever has been run so far."""
        df_report = pd.DataFrame(wm_report_rows)
        print("\n" + "=" * 72)
        print("WM REREFERENCING REPORT (per patient)")
        print("=" * 72)
        if len(df_report) == 0:
            print("(no patients processed)")
        else:
            # Compact print: status | n_wm_used | n_channels_used | error?
            for _, r in df_report.iterrows():
                status_tag = {
                    "ok":                  "  OK   ",
                    "ok-no-trials":        "OK-NOTR",
                    "ok-no-wm-grid":       "OK-GRID",
                    "no-neural-channels":  "NO-CHAN",
                    "error-reref":         "REREF✗ ",
                    "error-load":          "LOAD✗  ",
                    "error-other":         "ERR✗   ",
                    "":                    "??     ",
                }.get(r["status"], r["status"])
                line = (f"  {status_tag}  {r['patient_id']:<14}  "
                        f"in={r['n_channels_in']:>4}  "
                        f"neural={r['n_channels_neural']:>4}  "
                        f"unknown_dropped={r['n_channels_unknown_dropped']:>3}  "
                        f"used={r['n_channels_used']:>4}  "
                        f"wm={r['n_wm_used']:>3}")
                if r["error"]:
                    line += f"  err='{r['error'][:60]}'"
                print(line)
            n_ok   = (df_report["status"] == "ok").sum()
            n_fail = (df_report["status"].astype(str).str.startswith("error")).sum()
            print("-" * 72)
            print(f"Summary: {n_ok}/{len(df_report)} patients fully processed   "
                  f"{n_fail} failed   "
                  f"(total Unknown dropped: {df_report['n_channels_unknown_dropped'].sum()};   "
                  f"total WM used: {df_report['n_wm_used'].sum()})")

            # Persist (TSV; survives kernel restart and is reviewable in Excel/etc.)
            try:
                out_tsv = os.path.join(run_root_raw, "wm_reref_report.tsv")
                os.makedirs(os.path.dirname(out_tsv) or ".", exist_ok=True)
                df_report.to_csv(out_tsv, sep="\t", index=False)
                print(f"  saved -> {out_tsv}")
            except Exception as _e_save:
                print(f"  [warn] could not save report TSV: {_e_save}")


## Per-patient runs

One cell per patient so any single patient can be re-run on its own. Run the definition cell above once first. Re-running a patient replaces its row in the report; `wm_report()` at the bottom summarises whatever has been run.


### G-06 — MicroEPI (Geneva) · French · files written as `PAT_6854`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)

> **NOTES:** Response timing intact. Its `prep0` held byte-identical 05-08/05-12 copies of every table, so every trial was counted twice until the content-hash de-duplication went in.


In [ ]:
run_patients("G-06")




G-06


### G-04 — MicroEPI (Geneva) · English/French · files written as `PAT_6704`

**macro + micro** · 160 trials in 4 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), reading (53), audio (FR) (52), audio (EN) (1)

> **NOTES:** The one **English** trial inside the French auditory block (`auditory_naming_ENG`, trial 42) — a real event on the day, not corruption. It used to become its own condition folder because language suffixes were enumerated and `_eng` was missing; now normalised by prefix. **The stale `auditory_naming_eng/` output folders from the pre-fix run are still on disk and need deleting.** Its preset also carries the cohort's only non-empty `fake_trials`.


In [ ]:
run_patients("G-04")




G-04
  [paths] data_dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\MicroEPI-G-04\task_FBM\data_LM\exp4_Lora\prep
  [paths] prep_dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\MicroEPI-G-04\task_FBM\data_LM\exp4_Lora\prep0
  [microepi] 207 channels = 156 macros + 51 micros
  [PAT_6704] aux drop: 207 → 193 channels
  [PAT_6704] WM channels from TSV: 25 -> ['FOD2', 'FOD3', 'FOD4', 'FOD5', 'AD8', 'HAD7']...
  [PAT_6704] micros left raw (no micro_reref_anchor in preset)
[notch] PAT_6704  (z>=3.0)
  [notch] 50 Hz: z=30.1, Q=59.1 — notching
  [notch] 100 Hz: z=32.0, Q=145.4 — notching
  [notch] 150 Hz: z=80.4, Q=255.5 — notching
  [notch] 200 Hz: z=124.3, Q=371.1 — notching
  [notch] 250 Hz: z=55.2, Q=471.3 — notching
  [notch] 300 Hz: z=194.5, Q=500.0 — notching
  [notch] 350 Hz: z=34.0, Q=500.0 — notching
  [notch] 400 Hz: z=126.2, Q=500.0 — notching
  [notch] 450 Hz: z=82.0, Q=500.0 — notching
  [notch] 500 Hz: z=37.1, Q=500.0 — notching
  [trials] skipping PAT_67

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] PAT_6704 audio FOD1: 516 NaNs → filling
*[warn] PAT_6704 audio FOD2: 516 NaNs → filling
*[warn] PAT_6704 audio FOD3: 516 NaNs → filling
*[warn] PAT_6704 audio FOD4: 516 NaNs → filling
*[warn] PAT_6704 audio FOD5: 516 NaNs → filling
*[warn] PAT_6704 audio FOD6: 516 NaNs → filling
*[warn] PAT_6704 audio FPD1: 516 NaNs → filling
*[warn] PAT_6704 audio FPD2: 516 NaNs → filling
*[warn] PAT_6704 audio FPD3: 516 NaNs → filling
*[warn] PAT_6704 audio FPD4: 516 NaNs → filling
*[warn] PAT_6704 audio FPD5: 516 NaNs → filling
*[warn] PAT_6704 audio FPD6: 516 NaNs → filling
*[warn] PAT_6704 audio FPD7: 516 NaNs → filling
*[warn] PAT_6704 audio FPD8: 516 NaNs → filling
*[warn] PAT_6704 audio FPD9: 516 NaNs → filling
*[warn] PAT_6704 audio FPD10: 516 NaNs → filling
*[warn] PAT_6704 audio TPD1: 516 NaNs → filling
*[warn] PAT_6704 audio TPD2: 516 NaNs → filling
*[warn] PAT_6704 audio TPD3: 516 NaNs → filling
*[warn] PAT_6704 audio TPD4: 516 NaNs → filling
*[warn] PAT_6704 audio TPD5: 516 NaNs → 

### G-05 — MicroEPI (Geneva) · French · files written as `PAT_6684`

**macro + micro** · 157 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), reading (53), audio (FR) (50)

> **NOTES:** Nothing unusual — no duplicate trial files, response timing intact.


In [ ]:
run_patients("G-05")


### G-01 — MicroEPI (Geneva) · French · files written as `PAT_5515`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)

> **NOTES:** Control for today's `trial_end` repair: its response times were already correct, and rebuilding them from the log reproduced all 320 rows to within 1 sample (0.49 ms). That is what pinned the join and the 2048 Hz rate.


In [ ]:
run_patients("G-01")


### G-02 — MicroEPI (Geneva) · French · files written as `PAT_5533`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)

> **NOTES:** **`trial_end` was a flat 2 s on every trial** (4096 samples, zero variance in all three blocks). Its log spelled the column `response_timex`, so `lf_micromacro`'s `response_time` lookup missed and fell back to `offsets + fs*2.0`. Header corrected and `trial_end` rebuilt in place by `repair_trial_end_from_log.py` — no PD re-extraction needed. Trial counts drop (audio 53 -> 45) because the 5 s cap and IQR trimming finally have real durations to act on.


In [19]:
run_patients("G-02")




G-02
  [paths] data_dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\MicroEPI-G-02\task_otherlabs\Lora_FLM\prep
  [paths] prep_dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\MicroEPI-G-02\task_otherlabs\Lora_FLM\prep0
  [microepi] 180 channels = 180 macros + 0 micros
  [PAT_5533] aux drop: 180 → 169 channels
  [PAT_5533] WM channels from TSV: 28 -> ['FOD2', 'FOD4', 'FOD5', 'OFD4', 'CAD1', 'CAD8']...
  [PAT_5533] micros left raw (no micro_reref_anchor in preset)
[notch] PAT_5533  (z>=3.0)
  [notch] 50 Hz: z=1.5 — no significant peak, skipping
  [notch] 100 Hz: z=20.7, Q=151.8 — notching
  [notch] 150 Hz: z=8.4, Q=244.4 — notching
  [notch] 200 Hz: z=69.3, Q=360.1 — notching
  [notch] 250 Hz: z=45.3, Q=467.9 — notching
  [notch] 300 Hz: z=49.7, Q=500.0 — notching
  [notch] 350 Hz: z=31.8, Q=500.0 — notching
  [notch] 400 Hz: z=30.7, Q=500.0 — notching
  [notch] 450 Hz: z=26.4, Q=500.0 — notching
  [notch] 500 Hz: z=19.6, Q=500.0 — notching
  [trials] skipping PAT_

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] PAT_5533 audio FOD1: 516 NaNs → filling
*[warn] PAT_5533 audio FOD2: 516 NaNs → filling
*[warn] PAT_5533 audio FOD3: 516 NaNs → filling
*[warn] PAT_5533 audio FOD4: 516 NaNs → filling
*[warn] PAT_5533 audio FOD5: 516 NaNs → filling
*[warn] PAT_5533 audio FOD6: 516 NaNs → filling
*[warn] PAT_5533 audio FOD7: 516 NaNs → filling
*[warn] PAT_5533 audio FOD8: 516 NaNs → filling
*[warn] PAT_5533 audio FOD9: 516 NaNs → filling
*[warn] PAT_5533 audio FOD10: 516 NaNs → filling
*[warn] PAT_5533 audio FOD11: 516 NaNs → filling
*[warn] PAT_5533 audio FOD12: 516 NaNs → filling
*[warn] PAT_5533 audio FPD1: 516 NaNs → filling
*[warn] PAT_5533 audio FPD2: 516 NaNs → filling
*[warn] PAT_5533 audio FPD3: 516 NaNs → filling
*[warn] PAT_5533 audio FPD4: 516 NaNs → filling
*[warn] PAT_5533 audio FPD5: 516 NaNs → filling
*[warn] PAT_5533 audio FPD6: 516 NaNs → filling
*[warn] PAT_5533 audio FPD7: 516 NaNs → filling
*[warn] PAT_5533 audio FPD8: 516 NaNs → filling
*[warn] PAT_5533 audio FPD9: 516 NaNs 

{'pid_raw': 'G-02',
 'patient_id': 'PAT_5533',
 'status': 'ok',
 'n_channels_in': 180,
 'n_channels_neural': 169,
 'n_channels_unknown_dropped': 0,
 'n_channels_used': 0,
 'n_wm_used': 27,
 'wm_channels_used': 'CAD1|CAD11|CAD12|CAD8|CAD9|FOD2|FOD4|FOD5|HAD5|IAD1|IAD10|IAD15|IAD7|IMD10|IMD14|IMD16|IMD2|IPD12|IPD8|OFD4|PCD5|SMD1|SMD2|SMD4|SMD5|TOD2|TOD9',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### G-03 — MicroEPI (Geneva) · French · files written as `PAT_6619`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)

> **NOTES:** **Same flat-2 s failure as G-02**, same cause, same repair. Audio drops 53 -> 36, the largest loss in the cohort, because its real responses spread to 47 s. Also failed on memory in the first pass, so run it alone.


In [20]:
run_patients("G-03")




G-03
  [paths] data_dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\MicroEPI-G-03\task_FBM\exp3_Lora1_LM_CLA_CLV\prep
  [paths] prep_dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\MicroEPI-G-03\task_FBM\exp3_Lora1_LM_CLA_CLV\prep0
  [microepi] 158 channels = 158 macros + 0 micros
  [PAT_6619] aux drop: 158 → 144 channels
  [PAT_6619] WM channels from TSV: 17 -> ['FPD3', 'FPD5', 'FOD2', 'FOD14', 'OFAD1', 'CAD2']...
  [PAT_6619] micros left raw (no micro_reref_anchor in preset)
[notch] PAT_6619  (z>=3.0)
  [notch] 50 Hz: z=1.5 — no significant peak, skipping
  [notch] 100 Hz: z=11.1, Q=140.0 — notching
  [notch] 150 Hz: z=6.4, Q=255.6 — notching
  [notch] 200 Hz: z=59.7, Q=365.6 — notching
  [notch] 250 Hz: z=12.4, Q=462.9 — notching
  [notch] 300 Hz: z=44.9, Q=500.0 — notching
  [notch] 350 Hz: z=11.3, Q=500.0 — notching
  [notch] 400 Hz: z=31.6, Q=500.0 — notching
  [notch] 450 Hz: z=6.4, Q=500.0 — notching
  [notch] 500 Hz: z=7.0, Q=500.0 — notching
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] PAT_6619 audio FPD1: 516 NaNs → filling
*[warn] PAT_6619 audio FPD2: 516 NaNs → filling
*[warn] PAT_6619 audio FPD3: 516 NaNs → filling
*[warn] PAT_6619 audio FPD4: 516 NaNs → filling
*[warn] PAT_6619 audio FPD5: 516 NaNs → filling
*[warn] PAT_6619 audio FPD6: 516 NaNs → filling
*[warn] PAT_6619 audio FPD7: 516 NaNs → filling
*[warn] PAT_6619 audio FPD8: 516 NaNs → filling
*[warn] PAT_6619 audio FPD9: 516 NaNs → filling
*[warn] PAT_6619 audio FOD1: 516 NaNs → filling
*[warn] PAT_6619 audio FOD2: 516 NaNs → filling
*[warn] PAT_6619 audio FOD3: 516 NaNs → filling
*[warn] PAT_6619 audio FOD4: 516 NaNs → filling
*[warn] PAT_6619 audio FOD5: 516 NaNs → filling
*[warn] PAT_6619 audio FOD6: 516 NaNs → filling
*[warn] PAT_6619 audio FOD7: 516 NaNs → filling
*[warn] PAT_6619 audio FOD8: 516 NaNs → filling
*[warn] PAT_6619 audio FOD9: 516 NaNs → filling
*[warn] PAT_6619 audio FOD10: 516 NaNs → filling
*[warn] PAT_6619 audio FOD11: 516 NaNs → filling
*[warn] PAT_6619 audio FOD12: 516 NaNs 

{'pid_raw': 'G-03',
 'patient_id': 'PAT_6619',
 'status': 'ok',
 'n_channels_in': 158,
 'n_channels_neural': 144,
 'n_channels_unknown_dropped': 0,
 'n_channels_used': 0,
 'n_wm_used': 16,
 'wm_channels_used': 'CAD2|CAD9|FOD14|FOD2|FPD3|FPD5|IAD1|IAD10|IAD11|IAD7|IAD8|IAD9|IMD10|IMD5|IPD8|PHD7',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### EL030 — Bern · German

**depth electrodes (SEEG)** · 151 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (44)


In [ ]:
run_patients("EL030")


### EL033 — Bern · German

**depth electrodes (SEEG)** · 159 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: audio (DE) (53), picture (53), reading (53)


In [ ]:
run_patients("EL033")


### EL034 — Bern · German

**depth electrodes (SEEG)** · 159 trials in 3 block(s) · 5 manual bad channel(s) · 3 unique trial file(s)

Blocks: audio (DE) (53), reading (53), picture (53)


In [ ]:
run_patients("EL034")


### EL035 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 13 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL035")


### EL036 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 9 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)

> **NOTES:** Failed on memory in the first pass.


In [6]:
run_patients("EL036")




EL036
[LF 14:45:09] Patient: EL036 | Block: LM
[LF 14:45:09] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL036\task_FBM\data_LM\raw
[LF 14:45:09] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL036\task_FBM\data_LM\prep0
  [EL036] dropped 10 'Unknown' channels (no parcellation in TSV)
[notch] EL036  (z>=3.0)
  [notch] 50 Hz: z=3.7, Q=49.5 — notching
  [notch] 100 Hz: z=17.5, Q=122.7 — notching
  [notch] 150 Hz: z=29.1, Q=234.9 — notching
  [notch] 200 Hz: z=168.8, Q=365.3 — notching
  [notch] 250 Hz: z=155.7, Q=452.4 — notching
  [notch] 300 Hz: z=194.3, Q=500.0 — notching
  [notch] 350 Hz: z=76.2, Q=500.0 — notching
  [notch] 400 Hz: z=328.0, Q=500.0 — notching
  [notch] 450 Hz: z=18.0, Q=477.2 — notching
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] EL036 audio nH_L1: 645 NaNs → filling
*[warn] EL036 audio nH_L2: 645 NaNs → filling
*[warn] EL036 audio nH_L3: 645 NaNs → filling
*[warn] EL036 audio nH_L4: 645 NaNs → filling
*[warn] EL036 audio nH_L5: 645 NaNs → filling
*[warn] EL036 audio nH_L6: 645 NaNs → filling
*[warn] EL036 audio nH_L7: 645 NaNs → filling
*[warn] EL036 audio nH_L8: 645 NaNs → filling
*[warn] EL036 audio pfc_R1: 645 NaNs → filling
*[warn] EL036 audio pfc_R2: 645 NaNs → filling
*[warn] EL036 audio pfc_R3: 645 NaNs → filling
*[warn] EL036 audio pfc_R4: 645 NaNs → filling
*[warn] EL036 audio pfc_R7: 645 NaNs → filling
*[warn] EL036 audio pfc_R8: 645 NaNs → filling
*[warn] EL036 audio pfc_R9: 645 NaNs → filling
*[warn] EL036 audio A_R1: 645 NaNs → filling
*[warn] EL036 audio A_R2: 645 NaNs → filling
*[warn] EL036 audio A_R3: 645 NaNs → filling
*[warn] EL036 audio A_R4: 645 NaNs → filling
*[warn] EL036 audio A_R5: 645 NaNs → filling
*[warn] EL036 audio A_R6: 645 NaNs → filling
*[warn] EL036 audio A_R7: 645 NaNs

{'pid_raw': 'EL036',
 'patient_id': 'EL036',
 'status': 'ok',
 'n_channels_in': 108,
 'n_channels_neural': 72,
 'n_channels_unknown_dropped': 10,
 'n_channels_used': 0,
 'n_wm_used': 4,
 'wm_channels_used': 'PFCR5|PFCR6|PHGR6|PHGR7',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### EL037 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 39 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)

> **NOTES:** **39 manual bad channels, by far the most in the cohort.** Worth a look before trusting its coverage.


In [7]:
run_patients("EL037")




EL037
[LF 14:51:35] Patient: EL037 | Block: LM
[LF 14:51:35] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL037\task_FBM\data_LM\raw
[LF 14:51:35] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL037\task_FBM\data_LM\prep0
  [EL037] dropped 1 'Unknown' channels (no parcellation in TSV)
[notch] EL037  (z>=3.0)
  [notch] 50 Hz: z=14.2, Q=51.3 — notching
  [notch] 100 Hz: z=23.0, Q=134.9 — notching
  [notch] 150 Hz: z=87.7, Q=236.3 — notching
  [notch] 200 Hz: z=99.3, Q=354.2 — notching
  [notch] 250 Hz: z=111.4, Q=434.7 — notching
  [notch] 300 Hz: z=139.8, Q=500.0 — notching
  [notch] 350 Hz: z=30.4, Q=455.6 — notching
  [notch] 400 Hz: z=130.7, Q=500.0 — notching
  [notch] 450 Hz: z=15.4, Q=448.2 — notching
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] EL037 audio A_L1: 645 NaNs → filling
*[warn] EL037 audio A_L10: 645 NaNs → filling
*[warn] EL037 audio A_L11: 645 NaNs → filling
*[warn] EL037 audio A_L2: 645 NaNs → filling
*[warn] EL037 audio A_L3: 645 NaNs → filling
*[warn] EL037 audio A_L4: 645 NaNs → filling
*[warn] EL037 audio A_L5: 645 NaNs → filling
*[warn] EL037 audio A_L6: 645 NaNs → filling
*[warn] EL037 audio A_L7: 645 NaNs → filling
*[warn] EL037 audio A_L8: 645 NaNs → filling
*[warn] EL037 audio A_L9: 645 NaNs → filling
*[warn] EL037 audio A_R1: 645 NaNs → filling
**[warn] EL037 audio A_R11: 645 NaNs → filling
**[warn] EL037 audio A_R2: 645 NaNs → filling
*[warn] EL037 audio A_R3: 645 NaNs → filling
*[warn] EL037 audio A_R4: 645 NaNs → filling
*[warn] EL037 audio CinG_L14: 645 NaNs → filling
*[warn] EL037 audio CinG_L2: 645 NaNs → filling
*[warn] EL037 audio EKG-: 645 NaNs → filling
*[warn] EL037 audio EntG_L1: 645 NaNs → filling
*[warn] EL037 audio EntG_L10: 645 NaNs → filling
*[warn] EL037 audio EntG_L11: 645 NaN

{'pid_raw': 'EL037',
 'patient_id': 'EL037',
 'status': 'ok',
 'n_channels_in': 156,
 'n_channels_neural': 123,
 'n_channels_unknown_dropped': 1,
 'n_channels_used': 0,
 'n_wm_used': 10,
 'wm_channels_used': 'AHL8|AHR8|AR9|CINGL1|ENTGL6|OFL13|OFL4|OFL5|PHL12|PHR1',
 'wm_channels_excluded_as_bad': 'CINGL12|CINGL13|PHR7',
 'error': ''}

### EL038 — Bern · German

**depth electrodes (SEEG)** · 141 trials in 3 block(s) · 5 manual bad channel(s) · 3 unique trial file(s)

Blocks: reading (51), picture (51), audio (DE) (39)

> **NOTES:** Failed on memory in the first pass.


In [8]:
run_patients("EL038")




EL038
[LF 15:03:53] Patient: EL038 | Block: LM
[LF 15:03:53] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL038\task_FBM\data_LM\raw
[LF 15:03:53] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL038\task_FBM\data_LM\prep0
  [EL038] dropped 6 'Unknown' channels (no parcellation in TSV)
[notch] EL038  (z>=3.0)
  [notch] 50 Hz: z=6.9, Q=51.1 — notching
  [notch] 100 Hz: z=18.9, Q=133.0 — notching
  [notch] 150 Hz: z=62.3, Q=258.4 — notching
  [notch] 200 Hz: z=76.4, Q=352.8 — notching
  [notch] 250 Hz: z=83.7, Q=447.1 — notching
  [notch] 300 Hz: z=115.4, Q=500.0 — notching
  [notch] 350 Hz: z=194.0, Q=500.0 — notching
  [notch] 400 Hz: z=214.7, Q=500.0 — notching
  [notch] 450 Hz: z=5.8, Q=467.2 — notching
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] EL038 audio A_L1: 645 NaNs → filling
*[warn] EL038 audio A_L10: 645 NaNs → filling
*[warn] EL038 audio A_L11: 645 NaNs → filling
*[warn] EL038 audio A_L12: 645 NaNs → filling
*[warn] EL038 audio A_L2: 645 NaNs → filling
*[warn] EL038 audio A_L3: 645 NaNs → filling
**[warn] EL038 audio A_L8: 645 NaNs → filling
*[warn] EL038 audio A_L9: 645 NaNs → filling
*[warn] EL038 audio CinG_R1: 645 NaNs → filling
*[warn] EL038 audio LinG_R1: 645 NaNs → filling
*[warn] EL038 audio LinG_R10: 645 NaNs → filling
*[warn] EL038 audio LinG_R3: 645 NaNs → filling
*[warn] EL038 audio LinG_R4: 645 NaNs → filling
*[warn] EL038 audio LinG_R5: 645 NaNs → filling
*[warn] EL038 audio LinG_R6: 645 NaNs → filling
*[warn] EL038 audio LinG_R7: 645 NaNs → filling
*[warn] EL038 audio LinG_R8: 645 NaNs → filling
*[warn] EL038 audio OF_R1: 645 NaNs → filling
*[warn] EL038 audio OF_R14: 645 NaNs → filling
*[warn] EL038 audio OF_R16: 645 NaNs → filling
*[warn] EL038 audio OF_R2: 645 NaNs → filling
*[warn] EL038 audi

{'pid_raw': 'EL038',
 'patient_id': 'EL038',
 'status': 'ok',
 'n_channels_in': 148,
 'n_channels_neural': 91,
 'n_channels_unknown_dropped': 6,
 'n_channels_used': 0,
 'n_wm_used': 8,
 'wm_channels_used': 'LINGR2|LINGR9|OFR4|OFR5|PHL5|PHL6|PHL7|PHL8',
 'wm_channels_excluded_as_bad': 'CINGR2|STOR1',
 'error': ''}

### EL040 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 3 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL040")


### EL042 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL042")


### EL043 — Bern · German

**strip hemisphere override** · 106 trials in 2 block(s) · 0 manual bad channel(s) · 2 unique trial file(s)

Blocks: audio (DE) (53), reading (53)

> **NOTES:** Reading block missing. Strip-hemisphere override applies.


In [18]:
run_patients("EL043")




EL043
[LF 21:55:44] Patient: EL043 | Block: LM
[LF 21:55:44] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL043\task_FBM\data_LM\raw
[LF 21:55:44] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL043\task_FBM\data_LM\prep0
  [EL043] cropped to 260s..3710s (3450.0s of 3773.5s)
  [EL043] stripped _L#/_R# per cfg.STRIP_HEMI_PATIENTS  -> first few: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8']
  [EL043] dropped 11 'Unknown' channels (no parcellation in TSV)
[notch] EL043  (z>=3.0)
  [notch] 50 Hz: z=14.8, Q=45.7 — notching
  [notch] 100 Hz: z=29.6, Q=116.9 — notching
  [notch] 150 Hz: z=17.1, Q=139.9 — notching
  [notch] 200 Hz: z=9.7, Q=143.1 — notching
  [notch] 250 Hz: z=8.1, Q=167.9 — notching
  [notch] 300 Hz: z=8.2, Q=198.8 — notching
  [notch] 350 Hz: z=9.1, Q=246.5 — notching
  [notch] 400 Hz: z=9.4, Q=301.9 — notching
  [notch] 450 Hz: z=9.8, Q=325.5 — notching
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] EL043 audio A1: 645 NaNs → filling
*[warn] EL043 audio A2: 645 NaNs → filling
*[warn] EL043 audio A3: 645 NaNs → filling
*[warn] EL043 audio A4: 645 NaNs → filling
*[warn] EL043 audio A7: 645 NaNs → filling
*[warn] EL043 audio EKG-: 645 NaNs → filling
*[warn] EL043 audio Hip1: 645 NaNs → filling
*[warn] EL043 audio Hip2: 645 NaNs → filling
*[warn] EL043 audio Hip3: 645 NaNs → filling
*[warn] EL043 audio Hip7: 645 NaNs → filling
*[warn] EL043 audio IOG1: 645 NaNs → filling
*[warn] EL043 audio IOG2: 645 NaNs → filling
*[warn] EL043 audio IOG3: 645 NaNs → filling
*[warn] EL043 audio IOG4: 645 NaNs → filling
*[warn] EL043 audio IOG5: 645 NaNs → filling
*[warn] EL043 audio IOG6: 645 NaNs → filling
*[warn] EL043 audio IOG7: 645 NaNs → filling
*[warn] EL043 audio IOG8: 645 NaNs → filling
*[warn] EL043 audio ITG1: 645 NaNs → filling
*[warn] EL043 audio ITG10: 645 NaNs → filling
*[warn] EL043 audio ITG2: 645 NaNs → filling
*[warn] EL043 audio ITG3: 645 NaNs → filling
*[warn] EL043 audio 

{'pid_raw': 'EL043',
 'patient_id': 'EL043',
 'status': 'ok',
 'n_channels_in': 140,
 'n_channels_neural': 107,
 'n_channels_unknown_dropped': 11,
 'n_channels_used': 0,
 'n_wm_used': 21,
 'wm_channels_used': 'A5|A6|HIP4|HIP5|HIP6|ISMG4|ISMG6|ISMG7|ITG5|ITG6|LING1|LING2|LING3|LING4|SSMG13|SSMG2|SSMG3|SSMG5|SSMG6|STG1|STG5',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### EL044 — Bern · German

**grid (ECoG) · per-grid CAR · grid + depth** · 53 trials in 1 block(s) · 13 manual bad channel(s) · 1 unique trial file(s)

Blocks: audio (DE) (53)

> **NOTES:** Grid patient — WM reref falls back to per-grid CAR. Only the auditory block ran, and 8 contacts are still unlocalised.


In [ ]:
run_patients("EL044")


### EL045 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL045")


### EL046 — Bern · unlabelled

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 2 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)

> **NOTES:** **No BIDS_elec entry at all** — anatomy comes from `EL046_Lookup.xlsx` instead: 13 WM channels, 103 analysable contacts, MNI for all 178. The 2 contacts flagged `isOut` (pI_L1, pI_L4) sit outside the brain and are now registered as bad channels. Condition label carries no language suffix, but the raw filename says GER.


In [8]:
run_patients("EL046")




EL046
[LF 19:10:17] Patient: EL046 | Block: LM
[LF 19:10:17] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL046\task_FBM\data_LM\raw
[LF 19:10:17] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL046\task_FBM\data_LM\prep0
  [EL046] cropped to 21120s..23320s (2200.0s of 28802.5s)
[LF 19:12:40] [Unknown] No electrodes TSV matching: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\BIDS_elec\SEEG-BERN\sub-EL046\ieeg\*_electrodes.tsv
[LF 19:12:40]   [lookup] EL046_Lookup.xlsx: 152 plugged contacts
[LF 19:12:40] [WM] EL046: no BIDS TSV; using anatomy Lookup -> 13 WM channels
[LF 19:12:40] [WM] EL046: no BIDS TSV; using anatomy Lookup -> 13 WM channels
[notch] EL046  (z>=3.0)
  [notch] 50 Hz: z=20.8, Q=53.3 — notching
  [notch] 100 Hz: z=22.0, Q=142.2 — notching
  [notch] 150 Hz: z=53.0, Q=242.0 — notching
  [notch] 200 Hz: z=39.7, Q=347.1 — notching
  [notch] 250 Hz: z=95.8, Q=453.8 — notching
  [notch] 300 Hz: z=76.7, Q=500.0 — not

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] EL046 audio A_L1: 645 NaNs → filling
*[warn] EL046 audio A_L10: 645 NaNs → filling
*[warn] EL046 audio A_L11: 645 NaNs → filling
*[warn] EL046 audio A_L12: 645 NaNs → filling
*[warn] EL046 audio A_L2: 645 NaNs → filling
*[warn] EL046 audio A_L3: 645 NaNs → filling
*[warn] EL046 audio A_L4: 645 NaNs → filling
*[warn] EL046 audio A_L8: 645 NaNs → filling
*[warn] EL046 audio A_L9: 645 NaNs → filling
*[warn] EL046 audio EMG-: 645 NaNs → filling
*[warn] EL046 audio FOp_L1: 645 NaNs → filling
*[warn] EL046 audio FOp_L6: 645 NaNs → filling
*[warn] EL046 audio FOp_L7: 645 NaNs → filling
*[warn] EL046 audio Fp_L-5: 645 NaNs → filling
*[warn] EL046 audio Fp_L-6: 645 NaNs → filling
*[warn] EL046 audio Fp_L-7: 645 NaNs → filling
*[warn] EL046 audio Fp_L-8: 645 NaNs → filling
*[warn] EL046 audio Fp_L1: 645 NaNs → filling
*[warn] EL046 audio Fp_L10: 645 NaNs → filling
*[warn] EL046 audio Fp_L11: 645 NaNs → filling
*[warn] EL046 audio Fp_L12: 645 NaNs → filling
*[warn] EL046 audio Fp_L2: 645 N

{'pid_raw': 'EL046',
 'patient_id': 'EL046',
 'status': 'ok',
 'n_channels_in': 152,
 'n_channels_neural': 92,
 'n_channels_unknown_dropped': 0,
 'n_channels_used': 0,
 'n_wm_used': 9,
 'wm_channels_used': 'AHR7|AHR8|AIL6|AIL8|AIL9|FOPL2|PHL11|PHL7|PIL17',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### EL048 — Bern · unlabelled

**depth electrodes (SEEG)** · 158 trials in 1 block(s) · 7 manual bad channel(s) · 3 unique trial file(s)

Blocks: nan (158)

> **NOTES:** **Blocked — waiting on the behavioural file.** All 158 trials currently have `trial_end` and `resp_accuracy` = NaN, so every one is filtered out and the patient yields nothing. Anatomy is solved: `Lookup.xlsx` (NOT the patient-named `EL048_Lookup.xlsx`, whose `natus` column is empty) gives 16 WM channels and 7 out-of-brain contacts, now bad. **But it has no MNI coordinates in either workbook**, so even once it runs it cannot enter the recon, the glassbrains or MOBA's 3-D brain.


In [10]:
run_patients("EL048")




EL048
[LF 18:03:31] Patient: EL048 | Block: LM
[LF 18:03:31] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL048\task_FBM\data_LM\raw
[LF 18:03:31] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\EL048\task_FBM\data_LM\prep0
  [EL048] cropped to 4600s..8468s (3868.0s of 28802.8s)
[LF 18:05:34] [Unknown] No electrodes TSV matching: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\BIDS_elec\SEEG-BERN\sub-EL048\ieeg\*_electrodes.tsv
[LF 18:05:34]   [lookup] Lookup.xlsx: 79 plugged contacts
[LF 18:05:34] [WM] EL048: no BIDS TSV; using anatomy Lookup -> 16 WM channels
[LF 18:05:34] [WM] EL048: no BIDS TSV; using anatomy Lookup -> 16 WM channels
[notch] EL048  (z>=3.0)
  [notch] 50 Hz: z=20.1, Q=27.7 — notching
  [notch] 100 Hz: z=13.3, Q=38.9 — notching
  [notch] 150 Hz: z=24.7, Q=155.9 — notching
  [notch] 200 Hz: z=16.0, Q=102.5 — notching
  [notch] 250 Hz: z=8.6, Q=218.0 — notching
  [notch] 300 Hz: z=20.5, Q=199.3 — notching
  [no

C:\Users\artoni\AppData\Roaming\Python\Python311\site-packages\pandas\core\base.py:662: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)
C:\Users\artoni\AppData\Roaming\Python\Python311\site-packages\pandas\core\base.py:662: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)
C:\Users\artoni\AppData\Roaming\Python\Python311\site-packages\pandas\core\base.py:662: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


[error] EL048: [EL048 | nan] annotation error: 0 rows offset<=onset; 158 rows trial_end<offset


{'pid_raw': 'EL048',
 'patient_id': 'EL048',
 'status': 'error-other',
 'n_channels_in': 111,
 'n_channels_neural': 79,
 'n_channels_unknown_dropped': 0,
 'n_channels_used': 0,
 'n_wm_used': 16,
 'wm_channels_used': 'AHL4|AHL5|AHR6|AL5|AR4|ENTGR6|ENTGR7|ENTGR8|PHGR10|PHGR6|PHGR7|PHGR8|PHGR9|PHR6|PHR7|PHR8',
 'wm_channels_excluded_as_bad': '',
 'error': '[EL048 | nan] annotation error: 0 rows offset<=onset; 158 rows trial_end<offset'}

### PAT_3455 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 152 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (53), reading (51), auditory_naming (48)

> **NOTES:** Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [ ]:
run_patients("PAT_3455")


### PAT_2868 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (53), reading (53)

> **NOTES:** Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [ ]:
run_patients("PAT_2868")


### PAT_3066 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 154 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (50), reading (50)

> **NOTES:** Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [ ]:
run_patients("PAT_3066")


### PAT_3301 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 43 trials in 1 block(s) · 0 manual bad channel(s) · 1 unique trial file(s)

Blocks: picture (43)

> **NOTES:** Picture block only. Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [ ]:
run_patients("PAT_3301")


### PAT_3390 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (53), reading (53)

> **NOTES:** Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [ ]:
run_patients("PAT_3390")


### PAT_3415 — HUG (Geneva) · unlabelled

**grid + depth** · 160 trials in 3 block(s) · 17 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (53), reading (53)

> **NOTES:** Grid + depth: surface contacts protected by prefix, WM reref still runs on the depths. Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [ ]:
run_patients("PAT_3415")


### PAT_3965 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 51 trials in 1 block(s) · 5 manual bad channel(s) · 1 unique trial file(s)

Blocks: reading (51)

> **NOTES:** Picture block only. Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [9]:
run_patients("PAT_3965")




PAT_3965
[LF 17:43:46] Patient: PAT_3965 | Block: LM
[LF 17:43:46] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_3965\task_FBM\data_LM\raw
[LF 17:43:46] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_3965\task_FBM\data_LM\prep0
[LF 17:44:03] Loaded TRC: shape=(4423936, 233), fs=2048 Hz
  [PAT_3965] dropped 4 'Unknown' channels (no parcellation in TSV)
[notch] PAT_3965  (z>=3.0)
  [notch] 50 Hz: z=36.3, Q=58.6 — notching
  [notch] 100 Hz: z=10.7, Q=142.9 — notching
  [notch] 150 Hz: z=127.0, Q=249.7 — notching
  [notch] 200 Hz: z=8.3, Q=337.2 — notching
  [notch] 250 Hz: z=148.1, Q=450.5 — notching
  [notch] 300 Hz: z=3.2, Q=500.0 — notching
  [notch] 350 Hz: z=107.4, Q=500.0 — notching
  [notch] 400 Hz: z=8.9, Q=500.0 — notching
  [notch] 450 Hz: z=58.1, Q=500.0 — notching
  [notch] 500 Hz: z=1.8 — no significant peak, skipping
  reading: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] PAT_3965 reading cad1: 387 NaNs → filling
*[warn] PAT_3965 reading cad2: 387 NaNs → filling
*[warn] PAT_3965 reading cad3: 387 NaNs → filling
*[warn] PAT_3965 reading cad4: 387 NaNs → filling
*[warn] PAT_3965 reading cad5: 387 NaNs → filling
*[warn] PAT_3965 reading cad6: 387 NaNs → filling
*[warn] PAT_3965 reading cad8: 387 NaNs → filling
*[warn] PAT_3965 reading cad9: 387 NaNs → filling
*[warn] PAT_3965 reading cad10: 387 NaNs → filling
*[warn] PAT_3965 reading cad11: 387 NaNs → filling
*[warn] PAT_3965 reading cad12: 387 NaNs → filling
*[warn] PAT_3965 reading cmd2: 387 NaNs → filling
*[warn] PAT_3965 reading cmd4: 387 NaNs → filling
*[warn] PAT_3965 reading cmd5: 387 NaNs → filling
*[warn] PAT_3965 reading cmd6: 387 NaNs → filling
*[warn] PAT_3965 reading cmd7: 387 NaNs → filling
*[warn] PAT_3965 reading cmd8: 387 NaNs → filling
*[warn] PAT_3965 reading cmd10: 387 NaNs → filling
**[warn] PAT_3965 reading cpd4: 387 NaNs → filling
*[warn] PAT_3965 reading cpd6: 387 NaNs → fill

{'pid_raw': 'PAT_3965',
 'patient_id': 'PAT_3965',
 'status': 'ok',
 'n_channels_in': 233,
 'n_channels_neural': 222,
 'n_channels_unknown_dropped': 4,
 'n_channels_used': 0,
 'n_wm_used': 44,
 'wm_channels_used': 'AD9|CAD7|CAG2|CAG3|CAG5|CAG8|CMD12|CMD3|CMD9|CMG4|CMG7|CMG8|CMG9|CPD14|CPD15|CPD5|CPG10|CPG12|CPG3|CPG9|HAD7|HAG7|HPD5|HPG12|IAD1|IAD15|IAD16|IAD5|IAD6|IAD7|IAD8|IAG13|IAG16|IAG7|IMD11|IMD4|IMG16|IMG18|IMG3|IMG8|IMG9|POD1|POG1|POG3',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### PAT_3975 — HUG (Geneva) · German

**depth electrodes (SEEG)** · 150 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: audio (DE) (50), picture (50), reading (50)

> **NOTES:** Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [11]:
run_patients("PAT_3975")




PAT_3975
[LF 16:01:28] Patient: PAT_3975 | Block: LM
[LF 16:01:28] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_3975\task_FBM\data_LM\raw
[LF 16:01:28] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_3975\task_FBM\data_LM\prep0
[LF 16:01:39] Loaded TRC: shape=(3169920, 226), fs=2048 Hz
  [PAT_3975] dropped 8 'Unknown' channels (no parcellation in TSV)
[notch] PAT_3975  (z>=3.0)
  [notch] 50 Hz: z=0.4 — no significant peak, skipping
  [notch] 100 Hz: z=0.1 — no significant peak, skipping
  [notch] 150 Hz: z=3.9, Q=247.4 — notching
  [notch] 200 Hz: z=0.5 — no significant peak, skipping
  [notch] 250 Hz: z=1.7 — no significant peak, skipping
  [notch] 300 Hz: z=2.0 — no significant peak, skipping
  [notch] 350 Hz: z=1.8 — no significant peak, skipping
  [notch] 400 Hz: z=0.5 — no significant peak, skipping
  [notch] 450 Hz: z=14.7, Q=500.0 — notching
  [notch] 500 Hz: z=0.0 — no significant peak, skipping
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] PAT_3975 audio FOG2: 645 NaNs → filling
*[warn] PAT_3975 audio FOG3: 645 NaNs → filling
*[warn] PAT_3975 audio FOG4: 645 NaNs → filling
*[warn] PAT_3975 audio FOG5: 645 NaNs → filling
*[warn] PAT_3975 audio FOG6: 645 NaNs → filling
*[warn] PAT_3975 audio FOG7: 645 NaNs → filling
*[warn] PAT_3975 audio FOG11: 645 NaNs → filling
*[warn] PAT_3975 audio FOG12: 645 NaNs → filling
*[warn] PAT_3975 audio FOG13: 645 NaNs → filling
*[warn] PAT_3975 audio CPG1: 645 NaNs → filling
*[warn] PAT_3975 audio CPG2: 645 NaNs → filling
*[warn] PAT_3975 audio CPG3: 645 NaNs → filling
*[warn] PAT_3975 audio CPG4: 645 NaNs → filling
*[warn] PAT_3975 audio CPG5: 645 NaNs → filling
*[warn] PAT_3975 audio CPG6: 645 NaNs → filling
*[warn] PAT_3975 audio CPG7: 645 NaNs → filling
*[warn] PAT_3975 audio CPG8: 645 NaNs → filling
*[warn] PAT_3975 audio CPG9: 645 NaNs → filling
*[warn] PAT_3975 audio CPG11: 645 NaNs → filling
*[warn] PAT_3975 audio CPG13: 645 NaNs → filling
*[warn] PAT_3975 audio CPG14: 645 Na

{'pid_raw': 'PAT_3975',
 'patient_id': 'PAT_3975',
 'status': 'ok',
 'n_channels_in': 226,
 'n_channels_neural': 216,
 'n_channels_unknown_dropped': 8,
 'n_channels_used': 0,
 'n_wm_used': 48,
 'wm_channels_used': 'AD7|AG6|CPD6|CPD7|CPD8|CPG10|CPG12|FOD11|FOD12|FOD13|FOD14|FOD3|FOD5|FOD6|FOD9|FOG1|FOG10|FOG8|FOG9|HAD11|HAG5|HAG7|IAD11|IAG1|IAG2|IAG3|IAG4|IAG6|IMD12|IMD14|IMD15|IMD16|IMD17|IMD18|IMD7|IMG10|IMG11|IMG12|IMG13|IMG14|IMG15|IMG4|IMG5|IMG8|PHD11|PHD7|PHG4|TPD2',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### PAT_3780 — HUG (Geneva) · French

**depth electrodes (SEEG)** · 150 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (51), audio (FR) (50), reading (49)

> **NOTES:** Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [8]:
run_patients("PAT_3780")




PAT_3780
[LF 17:20:33] Patient: PAT_3780 | Block: LM
[LF 17:20:33] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_3780\task_FBM\data_LM\raw
[LF 17:20:33] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_3780\task_FBM\data_LM\prep0
[LF 17:20:39] Loaded TRC: shape=(2506752, 152), fs=2048 Hz
  [PAT_3780] dropped 5 'Unknown' channels (no parcellation in TSV)
[notch] PAT_3780  (z>=3.0)
  [notch] 50 Hz: z=0.5 — no significant peak, skipping
  [notch] 100 Hz: z=0.1 — no significant peak, skipping
  [notch] 150 Hz: z=1.2 — no significant peak, skipping
  [notch] 200 Hz: z=1.3 — no significant peak, skipping
  [notch] 250 Hz: z=4.7, Q=441.8 — notching
  [notch] 300 Hz: z=1.0 — no significant peak, skipping
  [notch] 350 Hz: z=8.2, Q=500.0 — notching
  [notch] 400 Hz: z=1.0 — no significant peak, skipping
  [notch] 450 Hz: z=10.7, Q=500.0 — notching
  [notch] 500 Hz: z=0.7 — no significant peak, skipping
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] PAT_3780 audio FMA1: 516 NaNs → filling
*[warn] PAT_3780 audio FMA2: 516 NaNs → filling
*[warn] PAT_3780 audio FMA3: 516 NaNs → filling
*[warn] PAT_3780 audio FMA4: 516 NaNs → filling
*[warn] PAT_3780 audio FMA5: 516 NaNs → filling
*[warn] PAT_3780 audio FMA8: 516 NaNs → filling
*[warn] PAT_3780 audio FMA9: 516 NaNs → filling
*[warn] PAT_3780 audio FMA10: 516 NaNs → filling
*[warn] PAT_3780 audio FMA11: 516 NaNs → filling
*[warn] PAT_3780 audio FMA12: 516 NaNs → filling
*[warn] PAT_3780 audio FAP1: 516 NaNs → filling
*[warn] PAT_3780 audio FAP2: 516 NaNs → filling
*[warn] PAT_3780 audio FAP3: 516 NaNs → filling
*[warn] PAT_3780 audio FAP4: 516 NaNs → filling
*[warn] PAT_3780 audio FAP5: 516 NaNs → filling
*[warn] PAT_3780 audio FAP6: 516 NaNs → filling
*[warn] PAT_3780 audio FAP7: 516 NaNs → filling
*[warn] PAT_3780 audio FAP8: 516 NaNs → filling
*[warn] PAT_3780 audio FLA1: 516 NaNs → filling
*[warn] PAT_3780 audio FLA2: 516 NaNs → filling
*[warn] PAT_3780 audio FLA3: 516 NaNs 

{'pid_raw': 'PAT_3780',
 'patient_id': 'PAT_3780',
 'status': 'ok',
 'n_channels_in': 152,
 'n_channels_neural': 136,
 'n_channels_unknown_dropped': 5,
 'n_channels_used': 0,
 'n_wm_used': 15,
 'wm_channels_used': 'FLP6|FMA6|FMA7|FMP8|FPS1|HAG4|IMG13|IMG17|IMG18|IMG4|IMG5|IMG6|IMG7|TOG2|TPG7',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

### PAT_6953 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 0 trials in 0 block(s) · 0 manual bad channel(s) · 0 unique trial file(s)

Blocks: none found

> **NOTES:** **No extracted trials on disk** — needs PD extraction before it can run. Could not start until 2026-08-14: `patient_ids` holds strings (`"PAT_3455"`) while the code wanted bare ints, which broke path building (`PAT_PAT_3455`) and, silently, the preset lookup.


In [8]:
run_patients("PAT_6953")




PAT_6953
[LF 18:52:12] Patient: PAT_6953 | Block: LM
[LF 18:52:12] Raw dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_6953\task_FBM\data_LM\raw
[LF 18:52:12] Prep dir: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\PAT_6953\task_FBM\data_LM\prep0
[LF 18:52:19] Loaded TRC: shape=(2631104, 193), fs=2048 Hz
[LF 18:52:23] [Unknown] No electrodes TSV matching: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\BIDS_elec\SEEG-HUG\sub-6953\ieeg\*_electrodes.tsv
[LF 18:52:23] [WM] PAT_6953: MANUAL override (temporary, no anatomy table) -> 9 requested: AD7, AD8, HPD4, HPD5, TPD5, TPD6, TPD7, IPD8, IPD9
[LF 18:52:23] [WM] PAT_6953: MANUAL override (temporary, no anatomy table) -> 9 requested: AD7, AD8, HPD4, HPD5, TPD5, TPD6, TPD7, IPD8, IPD9
[LF 18:52:23] [WM] PAT_6953: manual list matched 9/9 channels in the recording
  audio: *

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\functions\lf_ersp.py:395: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(mats, 0), axis=0)


[warn] PAT_6953 audio HAD1: 387 NaNs → filling
*[warn] PAT_6953 audio HAD2: 387 NaNs → filling
*[warn] PAT_6953 audio HAD3: 387 NaNs → filling
*[warn] PAT_6953 audio HAD4: 387 NaNs → filling
*[warn] PAT_6953 audio HAD5: 387 NaNs → filling
*[warn] PAT_6953 audio HAD6: 387 NaNs → filling
*[warn] PAT_6953 audio HAD7: 387 NaNs → filling
*[warn] PAT_6953 audio HAD8: 387 NaNs → filling
*[warn] PAT_6953 audio HAD9: 387 NaNs → filling
*[warn] PAT_6953 audio HAD10: 387 NaNs → filling
*[warn] PAT_6953 audio HPD1: 387 NaNs → filling
*[warn] PAT_6953 audio HPD2: 387 NaNs → filling
*[warn] PAT_6953 audio HPD3: 387 NaNs → filling
*[warn] PAT_6953 audio HPD6: 387 NaNs → filling
*[warn] PAT_6953 audio HPD7: 387 NaNs → filling
*[warn] PAT_6953 audio HPD8: 387 NaNs → filling
*[warn] PAT_6953 audio HPD9: 387 NaNs → filling
*[warn] PAT_6953 audio HPD10: 387 NaNs → filling
*[warn] PAT_6953 audio TMD1: 387 NaNs → filling
*[warn] PAT_6953 audio TMD2: 387 NaNs → filling
*[warn] PAT_6953 audio TMD3: 387 NaNs →

{'pid_raw': 'PAT_6953',
 'patient_id': 'PAT_6953',
 'status': 'ok',
 'n_channels_in': 193,
 'n_channels_neural': 182,
 'n_channels_unknown_dropped': 0,
 'n_channels_used': 0,
 'n_wm_used': 9,
 'wm_channels_used': 'AD7|AD8|HPD4|HPD5|IPD8|IPD9|TPD5|TPD6|TPD7',
 'wm_channels_excluded_as_bad': '',
 'error': ''}

## Report over everything run so far


In [ ]:
wm_report()
